In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, count, isnan, when
import matplotlib.pyplot as plt
import seaborn as sns

spark = SparkSession.builder \
    .appName("SilverDataEDA") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(" Spark session created")
print(f"Spark version: {spark.version}")

 Spark session created
Spark version: 3.5.0


In [19]:
import pandas as pd
import glob
import os

# Task 1: Load and basic stats of Silver Label Store data
print("🔍 Silver Label Store EDA - Task 1: Basic Overview")
print("=" * 60)

# Load all silver label store files (12 months)
silver_label_path = "/home/jovyan/scripts/datamart/silver/label_store/"
parquet_files = glob.glob(os.path.join(silver_label_path, "*.parquet"))
print(f"📁 Found {len(parquet_files)} parquet files")

# Load all data
df_list = []
for file in sorted(parquet_files):
    df_temp = pd.read_parquet(file)
    df_list.append(df_temp)
    print(f"   📄 {os.path.basename(file)}: {len(df_temp)} rows")

# Combine all data
df_all = pd.concat(df_list, ignore_index=True)
print(f"\n📊 Total combined data: {len(df_all)} rows")

# Basic info
print(f"\n📋 Schema Overview:")
print(f"   Columns: {list(df_all.columns)}")
print(f"   Data types:")
for col in df_all.columns:
    print(f"      {col}: {df_all[col].dtype}")

print(f"\n📈 Basic Statistics:")
print(f"   Total rows: {len(df_all):,}")
print(f"   Unique customers: {df_all['Customer_ID'].nunique()}")
print(f"   Unique loans: {df_all['loan_id'].nunique()}")
print(f"   Date range: {df_all['snapshot_date'].min()} to {df_all['snapshot_date'].max()}")

# Sample data
print(f"\n🔎 Sample Data (first 5 rows):")
print(df_all.head())

print(f"\n📋 Missing Values:")
print(df_all.isnull().sum())

print("\n✅ Task 1 Complete - Ready for next task!")

🔍 Silver Label Store EDA - Task 1: Basic Overview
📁 Found 24 parquet files
   📄 silver_label_store_2023_01_01.parquet: 530 rows
   📄 silver_label_store_2023_02_01.parquet: 1031 rows
   📄 silver_label_store_2023_03_01.parquet: 1537 rows
   📄 silver_label_store_2023_04_01.parquet: 2047 rows
   📄 silver_label_store_2023_05_01.parquet: 2568 rows
   📄 silver_label_store_2023_06_01.parquet: 3085 rows
   📄 silver_label_store_2023_07_01.parquet: 3556 rows
   📄 silver_label_store_2023_08_01.parquet: 4037 rows
   📄 silver_label_store_2023_09_01.parquet: 4491 rows
   📄 silver_label_store_2023_10_01.parquet: 4978 rows
   📄 silver_label_store_2023_11_01.parquet: 5469 rows
   📄 silver_label_store_2023_12_01.parquet: 5428 rows
   📄 silver_label_store_2024_01_01.parquet: 5412 rows
   📄 silver_label_store_2024_02_01.parquet: 5424 rows
   📄 silver_label_store_2024_03_01.parquet: 5425 rows
   📄 silver_label_store_2024_04_01.parquet: 5417 rows
   📄 silver_label_store_2024_05_01.parquet: 5391 rows
   📄 sil

In [20]:
# Task 1.5: Deep dive into missing values - especially first_missed_date
print("🔍 Silver Label Store EDA - Task 1.5: Missing Values Deep Dive")
print("=" * 70)

# 1. Overall missing value pattern
print("📊 1. Missing Values Overview:")
missing_summary = df_all.isnull().sum()
missing_pct = (missing_summary / len(df_all) * 100).round(2)
for col in missing_summary.index:
    if missing_summary[col] > 0:
        print(f"   {col}: {missing_summary[col]:,} missing ({missing_pct[col]}%)")

# 2. Focus on first_missed_date
print(f"\n🔍 2. first_missed_date Analysis:")
print(f"   Total records: {len(df_all):,}")
print(f"   Records with first_missed_date: {df_all['first_missed_date'].notna().sum():,}")
print(f"   Records without first_missed_date: {df_all['first_missed_date'].isna().sum():,}")

# 3. Correlation with other fields
print(f"\n📊 3. first_missed_date vs other risk indicators:")

# Check if missing first_missed_date correlates with no missed payments
no_missed_date = df_all['first_missed_date'].isna()
print(f"   Records without first_missed_date:")
print(f"      - DPD = 0: {df_all[no_missed_date]['dpd'].eq(0).sum():,}")
print(f"      - installments_missed = 0: {df_all[no_missed_date]['installments_missed'].eq(0).sum():,}")
print(f"      - is_current_delinquent = 0: {df_all[no_missed_date]['is_current_delinquent'].eq(0).sum():,}")

# Check records WITH first_missed_date  
has_missed_date = df_all['first_missed_date'].notna()
print(f"\n   Records WITH first_missed_date:")
print(f"      - DPD > 0: {df_all[has_missed_date]['dpd'].gt(0).sum():,}")
print(f"      - installments_missed > 0: {df_all[has_missed_date]['installments_missed'].gt(0).sum():,}")
print(f"      - is_current_delinquent = 1: {df_all[has_missed_date]['is_current_delinquent'].eq(1).sum():,}")

# 4. Sample records with and without first_missed_date
print(f"\n🔎 4. Sample Data Comparison:")
print("   Records WITHOUT first_missed_date (first 3):")
print(df_all[no_missed_date][['Customer_ID', 'dpd', 'installments_missed', 'is_current_delinquent', 'payment_performance']].head(3))

print("\n   Records WITH first_missed_date (first 3):")
print(df_all[has_missed_date][['Customer_ID', 'first_missed_date', 'dpd', 'installments_missed', 'is_current_delinquent', 'payment_performance']].head(3))

# 5. Hypothesis check
print(f"\n💡 5. Hypothesis Check:")
print("   Hypothesis: first_missed_date is NULL when customer never missed a payment")
never_missed = (df_all['dpd'] == 0) & (df_all['installments_missed'] == 0)
print(f"   Records with DPD=0 AND installments_missed=0: {never_missed.sum():,}")
print(f"   Of these, how many have NULL first_missed_date: {df_all[never_missed]['first_missed_date'].isna().sum():,}")

print("\n✅ Task 1.5 Complete - Missing values analysis done!")

🔍 Silver Label Store EDA - Task 1.5: Missing Values Deep Dive
📊 1. Missing Values Overview:
   first_missed_date: 83,952 missing (80.5%)
   payment_ratio: 11,974 missing (11.48%)

🔍 2. first_missed_date Analysis:
   Total records: 104,288
   Records with first_missed_date: 20,336
   Records without first_missed_date: 83,952

📊 3. first_missed_date vs other risk indicators:
   Records without first_missed_date:
      - DPD = 0: 83,952
      - installments_missed = 0: 83,952
      - is_current_delinquent = 0: 83,952

   Records WITH first_missed_date:
      - DPD > 0: 20,336
      - installments_missed > 0: 20,336
      - is_current_delinquent = 1: 20,336

🔎 4. Sample Data Comparison:
   Records WITHOUT first_missed_date (first 3):
  Customer_ID  dpd  installments_missed  is_current_delinquent  \
0  CUS_0x1037    0                    0                      0   
1  CUS_0x1069    0                    0                      0   
2  CUS_0x114a    0                    0                      0

In [21]:
# Task 2: Risk fields distribution analysis
print("🔍 Silver Label Store EDA - Task 2: Risk Fields Analysis")
print("=" * 60)

# 1. DPD (Days Past Due) Analysis
print("📊 1. DPD (Days Past Due) Distribution:")
print(f"   DPD stats: min={df_all['dpd'].min()}, max={df_all['dpd'].max()}, mean={df_all['dpd'].mean():.2f}")
print("   DPD value counts:")
print(df_all['dpd'].value_counts().head(10))

# 2. DPD Bucket Analysis  
print(f"\n📊 2. DPD Bucket Distribution:")
print(df_all['dpd_bucket'].value_counts())

# 3. Payment Performance Analysis
print(f"\n📊 3. Payment Performance Distribution:")
print(df_all['payment_performance'].value_counts())

# 4. Risk Flags Analysis
print(f"\n📊 4. Risk Flags Distribution:")
print(f"   is_high_risk: {df_all['is_high_risk'].value_counts().to_dict()}")
print(f"   is_current_delinquent: {df_all['is_current_delinquent'].value_counts().to_dict()}")
print(f"   is_severely_delinquent: {df_all['is_severely_delinquent'].value_counts().to_dict()}")
print(f"   early_warning_flag: {df_all['early_warning_flag'].value_counts().to_dict()}")

# 5. MOB (Months on Books) Analysis
print(f"\n📊 5. MOB (Months on Books) Distribution:")
print(df_all['mob'].value_counts().sort_index())

# 6. Payment Ratio Analysis
print(f"\n📊 6. Payment Ratio Distribution:")
print(f"   Payment ratio stats: min={df_all['payment_ratio'].min():.2f}, max={df_all['payment_ratio'].max():.2f}, mean={df_all['payment_ratio'].mean():.2f}")
print("   Payment ratio distribution (binned):")
payment_ratio_bins = pd.cut(df_all['payment_ratio'], bins=[-np.inf,0, 0.5, 0.8, 0.95, 1.0, 2.0], labels=["Not_Due", 'Poor(<0.5)', 'Fair(0.5-0.8)', 'Good(0.8-0.95)', 'Excellent(0.95-1.0)', 'Overpay(>1.0)'])
print(payment_ratio_bins.value_counts())

print("\n✅ Task 2 Complete - Ready for next task!")

🔍 Silver Label Store EDA - Task 2: Risk Fields Analysis
📊 1. DPD (Days Past Due) Distribution:
   DPD stats: min=0, max=306, mean=22.82
   DPD value counts:
dpd
0      83952
31      2385
61      2245
92      1707
153     1678
30      1580
122     1394
91       847
183      820
184      811
Name: count, dtype: int64

📊 2. DPD Bucket Distribution:
dpd_bucket
Current        83952
DPD_90_Plus    12834
DPD_61_90       2876
DPD_31_60       2753
DPD_1_30        1873
Name: count, dtype: int64

📊 3. Payment Performance Distribution:
payment_performance
Excellent     71978
No_Payment    20336
Not_Due       11974
Name: count, dtype: int64

📊 4. Risk Flags Distribution:
   is_high_risk: {0: 84245, 1: 20043}
   is_current_delinquent: {0: 83952, 1: 20336}
   is_severely_delinquent: {0: 91303, 1: 12985}
   early_warning_flag: {0: 101195, 1: 3093}

📊 5. MOB (Months on Books) Distribution:
mob
0     11974
1     11459
2     10971
3     10515
4     10022
5      9479
6      8974
7      8476
8      7985
9 

In [22]:
# Task 3: Customer-Level Aggregation and Risk Trajectory Analysis
print("🚀 Silver Label Store EDA - Task 3: Customer-Level Analysis")
print("=" * 70)

# 1. Customer Overview
print("📊 1. Customer Overview:")
unique_customers = df_all['Customer_ID'].nunique()
total_records = len(df_all)
avg_records_per_customer = total_records / unique_customers

print(f"   Total unique customers: {unique_customers:,}")
print(f"   Total records: {total_records:,}")
print(f"   Average records per customer: {avg_records_per_customer:.1f}")
print(f"   Max MOB in dataset: {df_all['mob'].max()}")

# 2. Customer Lifecycle Analysis
print(f"\n📈 2. Customer Lifecycle Analysis:")
customer_lifecycle = df_all.groupby('Customer_ID').agg({
    'mob': ['min', 'max', 'count'],
    'dpd': ['max', 'mean'],
    'is_current_delinquent': 'sum',
    'payment_performance': lambda x: x.mode().iloc[0] if not x.empty else 'Unknown'
}).round(2)

customer_lifecycle.columns = ['min_mob', 'max_mob', 'months_observed', 
                             'max_dpd_ever', 'avg_dpd', 'months_delinquent', 'dominant_performance']

print("   Customer lifecycle summary:")
print(f"   Average months observed per customer: {customer_lifecycle['months_observed'].mean():.1f}")
print(f"   Customers observed for 10+ months: {(customer_lifecycle['months_observed'] >= 10).sum():,}")
print(f"   Max DPD ever experienced distribution:")
print(customer_lifecycle['max_dpd_ever'].describe())

# 3. Customer Risk Transition Analysis
print(f"\n🔄 3. Customer Risk Transition Analysis:")

# Create customer risk profiles
risk_transitions = []
for customer in df_all['Customer_ID'].unique()[:1000]:  # Sample first 1000 customers for performance
    customer_data = df_all[df_all['Customer_ID'] == customer].sort_values('mob')
    if len(customer_data) >= 2:  # Need at least 2 months of data
        
        first_risk = customer_data.iloc[0]['dpd_bucket']
        last_risk = customer_data.iloc[-1]['dpd_bucket']
        months_observed = len(customer_data)
        ever_delinquent = (customer_data['is_current_delinquent'] > 0).any()
        max_dpd = customer_data['dpd'].max()
        
        risk_transitions.append({
            'Customer_ID': customer,
            'first_risk_bucket': first_risk,
            'last_risk_bucket': last_risk,
            'months_observed': months_observed,
            'ever_delinquent': ever_delinquent,
            'max_dpd_ever': max_dpd,
            'risk_worsened': first_risk == 'Current' and last_risk != 'Current',
            'risk_improved': first_risk != 'Current' and last_risk == 'Current'
        })

transitions_df = pd.DataFrame(risk_transitions)

print(f"   Analyzed {len(transitions_df)} customers with 2+ months of data:")
print(f"   Started good, ended bad (risk worsened): {transitions_df['risk_worsened'].sum():,}")
print(f"   Started bad, ended good (risk improved): {transitions_df['risk_improved'].sum():,}")
print(f"   Ever experienced delinquency: {transitions_df['ever_delinquent'].sum():,}")

# 4. MOB-based Risk Patterns
print(f"\n📅 4. MOB-based Risk Patterns:")
mob_risk_analysis = df_all.groupby('mob').agg({
    'is_current_delinquent': 'mean',
    'is_high_risk': 'mean', 
    'dpd': 'mean',
    'Customer_ID': 'count'
}).round(3)

mob_risk_analysis.columns = ['delinquency_rate', 'high_risk_rate', 'avg_dpd', 'customer_count']

print("   Risk rates by MOB:")
print(mob_risk_analysis)

# 5. Payment Behavior Consistency
print(f"\n💳 5. Payment Behavior Consistency Analysis:")
customer_payment_consistency = df_all.groupby('Customer_ID').agg({
    'payment_performance': lambda x: len(x.unique()),  # Number of different performance levels
    'payment_ratio': ['mean', 'std'],
    'mob': 'max'
}).round(3)

customer_payment_consistency.columns = ['performance_categories', 'avg_payment_ratio', 'payment_ratio_std', 'max_mob']

# Filter customers with sufficient history (3+ months)
experienced_customers = customer_payment_consistency[customer_payment_consistency['max_mob'] >= 3]

print(f"   Customers with 3+ months history: {len(experienced_customers):,}")
print(f"   Consistent performers (1 performance category): {(experienced_customers['performance_categories'] == 1).sum():,}")
print(f"   Variable performers (2+ categories): {(experienced_customers['performance_categories'] >= 2).sum():,}")

consistent_performers = experienced_customers[experienced_customers['performance_categories'] == 1]
print(f"   Average payment ratio of consistent performers: {consistent_performers['avg_payment_ratio'].mean():.3f}")

# 6. Early Warning Indicators for Gold Layer Design
print(f"\n⚠️ 6. Early Warning Indicators for Gold Layer:")

# Identify customers who became delinquent after starting good
early_warning_analysis = []
for customer in df_all['Customer_ID'].unique()[:500]:  # Sample for performance
    customer_data = df_all[df_all['Customer_ID'] == customer].sort_values('mob')
    
    if len(customer_data) >= 3:  # Need sufficient history
        # Check if customer started good but became bad
        first_month_good = customer_data.iloc[0]['dpd'] == 0
        became_delinquent = (customer_data['dpd'] > 0).any()
        
        if first_month_good and became_delinquent:
            # Find the month they first became delinquent
            first_delinquent_idx = customer_data[customer_data['dpd'] > 0].index[0]
            months_to_delinquency = customer_data.loc[first_delinquent_idx, 'mob']
            
            early_warning_analysis.append({
                'Customer_ID': customer,
                'months_to_first_delinquency': months_to_delinquency,
                'max_dpd_reached': customer_data['dpd'].max(),
                'recovered': (customer_data.iloc[-1]['dpd'] == 0) and (customer_data['dpd'].max() > 0)
            })

if early_warning_analysis:
    early_warning_df = pd.DataFrame(early_warning_analysis)
    print(f"   Customers who started good but became delinquent: {len(early_warning_df):,}")
    print(f"   Average months to first delinquency: {early_warning_df['months_to_first_delinquency'].mean():.1f}")
    print(f"   Customers who recovered after delinquency: {early_warning_df['recovered'].sum():,}")

print("\n✅ Task 3 Complete - Customer-level analysis done!")

🚀 Silver Label Store EDA - Task 3: Customer-Level Analysis
📊 1. Customer Overview:
   Total unique customers: 11,974
   Total records: 104,288
   Average records per customer: 8.7
   Max MOB in dataset: 10

📈 2. Customer Lifecycle Analysis:
   Customer lifecycle summary:
   Average months observed per customer: 8.7
   Customers observed for 10+ months: 7,472
   Max DPD ever experienced distribution:
count    11974.000000
mean        48.313763
std         89.811910
min          0.000000
25%          0.000000
50%          0.000000
75%         30.000000
max        306.000000
Name: max_dpd_ever, dtype: float64

🔄 3. Customer Risk Transition Analysis:
   Analyzed 1000 customers with 2+ months of data:
   Started good, ended bad (risk worsened): 286
   Started bad, ended good (risk improved): 0
   Ever experienced delinquency: 286

📅 4. MOB-based Risk Patterns:
   Risk rates by MOB:
     delinquency_rate  high_risk_rate  avg_dpd  customer_count
mob                                            

# Gold


In [23]:
import os
print(os.getcwd())


/home/jovyan/scripts


In [36]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob

print("🔍 Gold Layer EDA - Task 1: Data Loading")
print("=" * 60)

# Define base directories
gold_feature_dir = "/home/jovyan/scripts/datamart/gold/feature_store/"
gold_label_dir = "/home/jovyan/scripts/datamart/gold/label_store/"

# Function to load all Gold Feature Store files
def load_gold_features():
    """Load all gold feature store parquet files"""
    feature_files = glob.glob(f"{gold_feature_dir}*.parquet")
    
    if not feature_files:
        print("❌ No gold feature store files found!")
        return None
    
    print(f"📁 Found {len(feature_files)} gold feature files:")
    for file in sorted(feature_files):
        print(f"   - {file}")
    
    # Load and combine all feature files
    feature_dfs = []
    for file in sorted(feature_files):
        df = pd.read_parquet(file)
        feature_dfs.append(df)
        print(f"✅ Loaded {file}: {len(df)} rows")
    
    # Combine all dataframes
    df_features_combined = pd.concat(feature_dfs, ignore_index=True)
    print(f"\n📊 Combined Gold Features: {len(df_features_combined)} total rows")
    
    return df_features_combined

# Function to load all Gold Label Store files  
def load_gold_labels():
    """Load all gold label store parquet files"""
    label_files = glob.glob(f"{gold_label_dir}*.parquet")
    
    if not label_files:
        print("❌ No gold label store files found!")
        return None
    
    print(f"📁 Found {len(label_files)} gold label files:")
    for file in sorted(label_files):
        print(f"   - {file}")
    
    # Load and combine all label files
    label_dfs = []
    for file in sorted(label_files):
        df = pd.read_parquet(file)
        label_dfs.append(df)
        print(f"✅ Loaded {file}: {len(df)} rows")
    
    # Combine all dataframes
    df_labels_combined = pd.concat(label_dfs, ignore_index=True)
    print(f"\n📊 Combined Gold Labels: {len(df_labels_combined)} total rows")
    
    return df_labels_combined

# Load Gold Feature Store data
print("\n🚀 Loading Gold Feature Store...")
df_gold_features = load_gold_features()

# Load Gold Label Store data  
print("\n🚀 Loading Gold Label Store...")
df_gold_labels = load_gold_labels()

# Basic validation
if df_gold_features is not None and df_gold_labels is not None:
    print(f"\n✅ Gold Data Loading Summary:")
    print(f"   📊 Features: {len(df_gold_features)} rows × {len(df_gold_features.columns)} columns")
    print(f"   📊 Labels: {len(df_gold_labels)} rows × {len(df_gold_labels.columns)} columns")
    
    # Check for date range
    if 'snapshot_date' in df_gold_features.columns:
        feature_dates = pd.to_datetime(df_gold_features['snapshot_date'])
        print(f"   📅 Feature date range: {feature_dates.min()} to {feature_dates.max()}")
    
    if 'snapshot_date' in df_gold_labels.columns:
        label_dates = pd.to_datetime(df_gold_labels['snapshot_date']) 
        print(f"   📅 Label date range: {label_dates.min()} to {label_dates.max()}")
    
    # Check customer alignment
    if 'Customer_ID' in df_gold_features.columns and 'Customer_ID' in df_gold_labels.columns:
        feature_customers = set(df_gold_features['Customer_ID'].unique())
        label_customers = set(df_gold_labels['Customer_ID'].unique())
        
        print(f"   👥 Unique customers in features: {len(feature_customers)}")
        print(f"   👥 Unique customers in labels: {len(label_customers)}")
        print(f"   🔗 Customer overlap: {len(feature_customers & label_customers)}")
        
        if len(feature_customers - label_customers) > 0:
            print(f"   ⚠️  Customers in features but not labels: {len(feature_customers - label_customers)}")
        if len(label_customers - feature_customers) > 0:
            print(f"   ⚠️  Customers in labels but not features: {len(label_customers - feature_customers)}")

print("\n✅ Task 1 Complete - Gold data loaded successfully!")

# Make data available for next task
print(f"\n📋 Available DataFrames:")
print(f"   - df_gold_features: {len(df_gold_features) if df_gold_features is not None else 0} rows")
print(f"   - df_gold_labels: {len(df_gold_labels) if df_gold_labels is not None else 0} rows")

🔍 Gold Layer EDA - Task 1: Data Loading

🚀 Loading Gold Feature Store...
📁 Found 24 gold feature files:
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_02_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_03_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_04_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_05_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_06_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_07_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_08_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_store_2023_09_01.parquet
   - /home/jovyan/scripts/datamart/gold/feature_store/gold_feature_s

In [37]:
print("🔍 Gold Layer EDA - Task 2: Customer ID Alignment Check")
print("=" * 70)

# Check basic customer alignment by snapshot date
print("📊 1. Customer Count by Snapshot Date:")
print("-" * 40)

# Features customer count by date
feature_customer_counts = df_gold_features.groupby('snapshot_date')['Customer_ID'].nunique().sort_index()
print("Features by date:")
for date, count in feature_customer_counts.items():
    print(f"   {date}: {count} customers")

print()

# Labels customer count by date  
label_customer_counts = df_gold_labels.groupby('snapshot_date')['Customer_ID'].nunique().sort_index()
print("Labels by date:")
for date, count in label_customer_counts.items():
    print(f"   {date}: {count} customers")

print("\n" + "=" * 50)

# Check for perfect alignment by date
print("🔗 2. Customer ID Alignment by Date:")
print("-" * 40)

# Convert to datetime and handle any NaN values
df_gold_features['snapshot_date'] = pd.to_datetime(df_gold_features['snapshot_date'])
df_gold_labels['snapshot_date'] = pd.to_datetime(df_gold_labels['snapshot_date'])

# Get unique dates, excluding any NaN values
feature_dates = set(df_gold_features['snapshot_date'].dropna().unique())
label_dates = set(df_gold_labels['snapshot_date'].dropna().unique())
all_dates = sorted(feature_dates | label_dates)

alignment_results = []
for date in all_dates:
    # Get customers for this date
    feature_customers = set(df_gold_features[df_gold_features['snapshot_date'] == date]['Customer_ID'].unique())
    label_customers = set(df_gold_labels[df_gold_labels['snapshot_date'] == date]['Customer_ID'].unique())
    
    # Calculate alignment metrics
    overlap = feature_customers & label_customers
    only_in_features = feature_customers - label_customers
    only_in_labels = label_customers - feature_customers
    
    alignment_results.append({
        'date': date,
        'feature_customers': len(feature_customers),
        'label_customers': len(label_customers),
        'overlap': len(overlap),
        'only_features': len(only_in_features),
        'only_labels': len(only_in_labels),
        'perfect_match': len(only_in_features) == 0 and len(only_in_labels) == 0
    })
    
    # Print results for this date
    print(f"📅 {date}:")
    print(f"   Features: {len(feature_customers)} | Labels: {len(label_customers)} | Overlap: {len(overlap)}")
    
    if len(only_in_features) > 0:
        print(f"   ⚠️  {len(only_in_features)} customers only in features")
    if len(only_in_labels) > 0:
        print(f"   ⚠️  {len(only_in_labels)} customers only in labels")
    if len(only_in_features) == 0 and len(only_in_labels) == 0:
        print(f"   ✅ Perfect alignment!")
    print()

print("=" * 50)

# Summary statistics
print("📈 3. Alignment Summary:")
print("-" * 30)

perfect_matches = sum(1 for result in alignment_results if result['perfect_match'])
total_dates = len(alignment_results)

print(f"📊 Total snapshot dates analyzed: {total_dates}")
print(f"✅ Dates with perfect customer alignment: {perfect_matches}")
print(f"⚠️  Dates with misalignment: {total_dates - perfect_matches}")

if perfect_matches == total_dates:
    print("\n🎉 SUCCESS: All dates have perfect customer alignment!")
else:
    print(f"\n⚠️  WARNING: {total_dates - perfect_matches} dates have customer misalignment")
    
    # Show problematic dates
    print("\n🔍 Problematic dates details:")
    for result in alignment_results:
        if not result['perfect_match']:
            print(f"   📅 {result['date']}: Features={result['feature_customers']}, Labels={result['label_customers']}")
            if result['only_features'] > 0:
                print(f"      → {result['only_features']} customers missing in labels")
            if result['only_labels'] > 0:
                print(f"      → {result['only_labels']} customers missing in features")

print("\n" + "=" * 50)

# Check for data consistency across time
print("📊 4. Customer Journey Consistency:")
print("-" * 40)

# Find customers who appear in all months vs some months
all_feature_customers = set(df_gold_features['Customer_ID'].unique())
all_label_customers = set(df_gold_labels['Customer_ID'].unique()) 

print(f"🔍 Overall customer universe:")
print(f"   Total unique customers in features: {len(all_feature_customers)}")
print(f"   Total unique customers in labels: {len(all_label_customers)}")
print(f"   Customers in both: {len(all_feature_customers & all_label_customers)}")

# Check customer consistency across months for features
customer_month_counts_features = df_gold_features['Customer_ID'].value_counts()
print(f"\n📈 Customer appearance frequency in Features:")
print(f"   Customers appearing in all 12 months: {sum(customer_month_counts_features == 12)}")
print(f"   Customers appearing in 10+ months: {sum(customer_month_counts_features >= 10)}")
print(f"   Customers appearing in 5+ months: {sum(customer_month_counts_features >= 5)}")
print(f"   Customers appearing in 1 month only: {sum(customer_month_counts_features == 1)}")

# Check customer consistency across months for labels  
customer_month_counts_labels = df_gold_labels['Customer_ID'].value_counts()
print(f"\n📈 Customer appearance frequency in Labels:")
print(f"   Customers appearing in all 12 months: {sum(customer_month_counts_labels == 12)}")
print(f"   Customers appearing in 10+ months: {sum(customer_month_counts_labels >= 10)}")
print(f"   Customers appearing in 5+ months: {sum(customer_month_counts_labels >= 5)}")
print(f"   Customers appearing in 1 month only: {sum(customer_month_counts_labels == 1)}")

print("\n✅ Task 2 Complete - Customer ID alignment analysis done!")

🔍 Gold Layer EDA - Task 2: Customer ID Alignment Check
📊 1. Customer Count by Snapshot Date:
----------------------------------------
Features by date:
   2023-01-01: 530 customers
   2023-02-01: 501 customers
   2023-03-01: 506 customers
   2023-04-01: 510 customers
   2023-05-01: 521 customers
   2023-06-01: 517 customers
   2023-07-01: 471 customers
   2023-08-01: 481 customers
   2023-09-01: 454 customers
   2023-10-01: 487 customers
   2023-11-01: 491 customers
   2023-12-01: 489 customers
   2024-01-01: 485 customers
   2024-02-01: 518 customers
   2024-03-01: 511 customers
   2024-04-01: 513 customers
   2024-05-01: 491 customers
   2024-06-01: 498 customers
   2024-07-01: 505 customers
   2024-08-01: 543 customers
   2024-09-01: 493 customers
   2024-10-01: 456 customers
   2024-11-01: 488 customers
   2024-12-01: 515 customers

Labels by date:
   2023-01-01: 530 customers
   2023-02-01: 1031 customers
   2023-03-01: 1537 customers
   2023-04-01: 2047 customers
   2023-05-01: 2

In [38]:
print("🔍 Gold Label Store Debug Investigation")
print("=" * 60)

# Check what's actually in the label data
print("📊 1. Label Data Structure Investigation:")
print("-" * 45)

# Look at a few sample records
print("🔍 Sample of Gold Labels (first 10 rows):")
print(df_gold_labels[['Customer_ID', 'snapshot_date', 'target']].head(10))

print("\n🔍 Sample records for specific dates:")
sample_dates = ['2023-01-01', '2023-02-01', '2023-03-01']

for date in sample_dates:
    date_data = df_gold_labels[df_gold_labels['snapshot_date'] == date]
    print(f"\n📅 {date}: {len(date_data)} records")
    if len(date_data) > 0:
        print("   Sample Customer_IDs (first 5):")
        print("  ", list(date_data['Customer_ID'].head().values))
        print(f"   Target distribution: {date_data['target'].value_counts().to_dict()}")

print("\n" + "=" * 50)

# Check for duplicate customers within the same date
print("📊 2. Duplicate Customer Check:")
print("-" * 35)

duplicates_by_date = {}
for date in df_gold_labels['snapshot_date'].unique():
    if pd.notna(date):
        date_data = df_gold_labels[df_gold_labels['snapshot_date'] == date]
        duplicate_customers = date_data['Customer_ID'].value_counts()
        duplicates = duplicate_customers[duplicate_customers > 1]
        
        duplicates_by_date[date] = len(duplicates)
        
        if len(duplicates) > 0:
            print(f"📅 {date}: {len(duplicates)} customers appear multiple times")
            print(f"   Example duplicates: {list(duplicates.head(3).index)}")
        else:
            print(f"📅 {date}: ✅ No duplicate customers")

print("\n" + "=" * 50)

# Check the columns in label data
print("📊 3. Label Schema Analysis:")
print("-" * 30)

print(f"🔍 Total columns in Gold Labels: {len(df_gold_labels.columns)}")
print("📋 Column list:")
for i, col in enumerate(df_gold_labels.columns):
    print(f"   {i+1:2d}. {col}")

print(f"\n📊 Key column info:")
print(f"   snapshot_date type: {df_gold_labels['snapshot_date'].dtype}")
print(f"   Customer_ID type: {df_gold_labels['Customer_ID'].dtype}")
if 'target' in df_gold_labels.columns:
    print(f"   target type: {df_gold_labels['target'].dtype}")
    print(f"   target values: {df_gold_labels['target'].unique()}")

print("\n" + "=" * 50)

# Investigation: Why are labels accumulating?
print("📊 4. Accumulation Pattern Investigation:")
print("-" * 45)

print("🔍 Customer ID patterns across dates:")
dates_sorted = sorted(df_gold_labels['snapshot_date'].dropna().unique())

prev_customers = set()
for date in dates_sorted[:4]:  # Check first 4 months
    current_customers = set(df_gold_labels[df_gold_labels['snapshot_date'] == date]['Customer_ID'].unique())
    
    if prev_customers:
        overlap = len(current_customers & prev_customers)
        new_customers = len(current_customers - prev_customers)
        print(f"📅 {date}: {len(current_customers)} total | {overlap} returning | {new_customers} new")
    else:
        print(f"📅 {date}: {len(current_customers)} customers (first month)")
    
    prev_customers = current_customers

print("\n✅ Debug investigation complete!")

🔍 Gold Label Store Debug Investigation
📊 1. Label Data Structure Investigation:
---------------------------------------------
🔍 Sample of Gold Labels (first 10 rows):
  Customer_ID snapshot_date  target
0  CUS_0x1037    2023-01-01       0
1  CUS_0x1069    2023-01-01       0
2  CUS_0x114a    2023-01-01       0
3  CUS_0x1184    2023-01-01       0
4  CUS_0x1297    2023-01-01       0
5  CUS_0x12fb    2023-01-01       0
6  CUS_0x1325    2023-01-01       0
7  CUS_0x1341    2023-01-01       0
8  CUS_0x1375    2023-01-01       0
9  CUS_0x13a8    2023-01-01       0

🔍 Sample records for specific dates:

📅 2023-01-01: 530 records
   Sample Customer_IDs (first 5):
   ['CUS_0x1037', 'CUS_0x1069', 'CUS_0x114a', 'CUS_0x1184', 'CUS_0x1297']
   Target distribution: {0: 530}

📅 2023-02-01: 1031 records
   Sample Customer_IDs (first 5):
   ['CUS_0x1037', 'CUS_0x1069', 'CUS_0x10aa', 'CUS_0x113e', 'CUS_0x1140']
   Target distribution: {0: 1006, 1: 25}

📅 2023-03-01: 1537 records
   Sample Customer_IDs (fi

In [39]:
import matplotlib.pyplot as plt
import seaborn as sns

print("🔍 Gold Layer EDA - Task 3: Feature Distribution Analysis")
print("=" * 70)

# Focus on one snapshot date for cleaner analysis
analysis_date = '2023-12-01'  # Latest month with most customers
df_features_sample = df_gold_features[df_gold_features['snapshot_date'] == analysis_date].copy()
df_labels_sample = df_gold_labels[df_gold_labels['snapshot_date'] == analysis_date].copy()

print(f"📊 Analysis focus: {analysis_date}")
print(f"   Features: {len(df_features_sample)} customers")
print(f"   Labels: {len(df_labels_sample)} customers")

# 1. Feature Types Analysis
print("\n📋 1. Feature Categories Overview:")
print("-" * 40)

# Categorize features
feature_categories = {
    'clickstream': [col for col in df_features_sample.columns if col.startswith('fe_')],
    'demographic': ['Age', 'age_group', 'occupation_category'],
    'clickstream_derived': ['total_clickstream_activity', 'avg_clickstream_activity', 'is_high_activity_user'],
    'financial_basic': ['Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 
                       'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
                       'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 
                       'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month',
                       'Amount_invested_monthly', 'Monthly_Balance'],
    'financial_derived': ['debt_to_income_ratio', 'monthly_debt_service_ratio', 'investment_rate',
                         'credit_utilization_category', 'financial_risk_score']
}

for category, columns in feature_categories.items():
    existing_columns = [col for col in columns if col in df_features_sample.columns]
    print(f"   {category}: {len(existing_columns)} features")
    if len(existing_columns) != len(columns):
        missing = set(columns) - set(existing_columns)
        print(f"      Missing: {missing}")

# 2. Numerical Features Analysis
print("\n📊 2. Numerical Features Statistics:")
print("-" * 40)

# Get numerical columns (excluding ID and date)
numerical_cols = df_features_sample.select_dtypes(include=[np.number]).columns.tolist()
if 'Customer_ID' in numerical_cols:
    numerical_cols.remove('Customer_ID')

print(f"📈 Found {len(numerical_cols)} numerical features")

# Basic statistics for key numerical features
key_numerical = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt', 
                'Credit_Utilization_Ratio', 'financial_risk_score']
key_numerical = [col for col in key_numerical if col in numerical_cols]

if key_numerical:
    print("\n🔍 Key numerical features summary:")
    summary_stats = df_features_sample[key_numerical].describe()
    print(summary_stats.round(2))

# 3. Clickstream Features Analysis
print("\n📊 3. Clickstream Features (fe_1 to fe_20):")
print("-" * 45)

clickstream_cols = [col for col in df_features_sample.columns if col.startswith('fe_')]
if clickstream_cols:
    print(f"📈 Analyzing {len(clickstream_cols)} clickstream features")
    
    # Summary statistics
    clickstream_stats = df_features_sample[clickstream_cols].describe()
    print("\n🔍 Clickstream features summary:")
    print(f"   Mean range: {clickstream_stats.loc['mean'].min():.2f} to {clickstream_stats.loc['mean'].max():.2f}")
    print(f"   Std range: {clickstream_stats.loc['std'].min():.2f} to {clickstream_stats.loc['std'].max():.2f}")
    print(f"   Min values: {clickstream_stats.loc['min'].min():.0f} to {clickstream_stats.loc['min'].max():.0f}")
    print(f"   Max values: {clickstream_stats.loc['max'].min():.0f} to {clickstream_stats.loc['max'].max():.0f}")
    
    # Check for zero variance features
    zero_var_features = []
    for col in clickstream_cols:
        if df_features_sample[col].std() == 0:
            zero_var_features.append(col)
    
    if zero_var_features:
        print(f"   ⚠️  Zero variance features: {zero_var_features}")
    else:
        print(f"   ✅ All clickstream features have variance")

# 4. Categorical Features Analysis
print("\n📊 4. Categorical Features Analysis:")
print("-" * 40)

categorical_cols = df_features_sample.select_dtypes(include=['object']).columns.tolist()
if 'Customer_ID' in categorical_cols:
    categorical_cols.remove('Customer_ID')

print(f"📈 Found {len(categorical_cols)} categorical features")

for col in categorical_cols:
    unique_vals = df_features_sample[col].nunique()
    print(f"   {col}: {unique_vals} unique values")
    
    # Show value counts for features with reasonable number of categories
    if unique_vals <= 10:
        value_counts = df_features_sample[col].value_counts()
        print(f"      Top values: {dict(value_counts.head(3))}")

# 5. Missing Values Analysis
print("\n📊 5. Missing Values Analysis:")
print("-" * 35)

missing_analysis = df_features_sample.isnull().sum()
missing_features = missing_analysis[missing_analysis > 0]

if len(missing_features) > 0:
    print("⚠️  Features with missing values:")
    for feature, count in missing_features.items():
        pct = (count / len(df_features_sample)) * 100
        print(f"   {feature}: {count} missing ({pct:.1f}%)")
else:
    print("✅ No missing values in feature data!")

# 6. Data Quality Checks
print("\n📊 6. Data Quality Checks:")
print("-" * 30)

# Check for duplicate customers
duplicates = df_features_sample['Customer_ID'].duplicated().sum()
print(f"   Duplicate customers: {duplicates}")

# Check for infinite values
inf_checks = {}
for col in numerical_cols:
    inf_count = np.isinf(df_features_sample[col]).sum()
    if inf_count > 0:
        inf_checks[col] = inf_count

if inf_checks:
    print("⚠️  Features with infinite values:")
    for feature, count in inf_checks.items():
        print(f"   {feature}: {count} infinite values")
else:
    print("✅ No infinite values detected!")

# Check for negative values where they shouldn't be
negative_checks = ['Annual_Income', 'Monthly_Inhand_Salary', 'Age']
negative_issues = {}
for col in negative_checks:
    if col in df_features_sample.columns:
        negative_count = (df_features_sample[col] < 0).sum()
        if negative_count > 0:
            negative_issues[col] = negative_count

if negative_issues:
    print("⚠️  Features with unexpected negative values:")
    for feature, count in negative_issues.items():
        print(f"   {feature}: {count} negative values")
else:
    print("✅ No unexpected negative values!")

print(f"\n✅ Task 3 Complete - Feature distribution analysis done!")
print(f"📋 Summary: {len(df_features_sample)} customers, {len(df_features_sample.columns)-2} features analyzed")

🔍 Gold Layer EDA - Task 3: Feature Distribution Analysis
📊 Analysis focus: 2023-12-01
   Features: 489 customers
   Labels: 5428 customers

📋 1. Feature Categories Overview:
----------------------------------------
   clickstream: 0 features
   demographic: 3 features
   clickstream_derived: 0 features
      Missing: {'total_clickstream_activity', 'is_high_activity_user', 'avg_clickstream_activity'}
   financial_basic: 16 features
   financial_derived: 5 features

📊 2. Numerical Features Statistics:
----------------------------------------
📈 Found 20 numerical features

🔍 Key numerical features summary:
         Age  Annual_Income  Monthly_Inhand_Salary  Outstanding_Debt  \
count  477.0         478.00                 476.00            458.00   
mean    33.6       45782.43                3719.59           1228.97   
std     11.0       34659.74                2792.25            862.08   
min     14.0        7171.26                 406.57              0.95   
25%     24.0       17938.08  

In [40]:
print("🔍 Gold Layer EDA - Task 3: Understanding Gold Data Structure & Feature Analysis")
print("=" * 80)

# Understand the data structure correctly
print("📊 1. Gold Layer Data Structure Understanding:")
print("-" * 50)

print("🧠 Hypothesis: Features = Monthly Active Customers, Labels = Cumulative Customer History")
print("\n🔍 Let's validate this understanding:")

# Analyze the business logic
sample_dates = ['2023-01-01', '2023-03-01', '2023-06-01', '2023-12-01']

for date in sample_dates:
    features_data = df_gold_features[df_gold_features['snapshot_date'] == date]
    labels_data = df_gold_labels[df_gold_labels['snapshot_date'] == date]
    
    print(f"\n📅 {date}:")
    print(f"   Features: {len(features_data)} customers (monthly active)")
    print(f"   Labels: {len(labels_data)} customers (cumulative history)")
    
    if len(features_data) > 0 and len(labels_data) > 0:
        # Check customer overlap
        feature_customers = set(features_data['Customer_ID'])
        label_customers = set(labels_data['Customer_ID'])
        overlap = feature_customers & label_customers
        
        print(f"   Overlap: {len(overlap)} customers")
        print(f"   Features only: {len(feature_customers - label_customers)} customers")
        print(f"   Labels only: {len(label_customers - feature_customers)} customers")
        
        # Check target distribution in labels
        if 'target' in labels_data.columns:
            target_dist = labels_data['target'].value_counts().sort_index()
            high_risk = target_dist.get(1.0, 0)
            low_risk = target_dist.get(0.0, 0)
            total_valid = high_risk + low_risk
            
            if total_valid > 0:
                print(f"   Target distribution: {high_risk}/{total_valid} high risk ({high_risk/total_valid:.1%})")

print("\n" + "=" * 60)

# Focus on a specific month for modeling analysis
print("📊 2. Modeling-Ready Analysis for June 2023:")
print("-" * 45)

analysis_date = '2023-06-01'
features_june = df_gold_features[df_gold_features['snapshot_date'] == analysis_date].copy()
labels_june = df_gold_labels[df_gold_labels['snapshot_date'] == analysis_date].copy()

print(f"🎯 Analysis Date: {analysis_date}")
print(f"   Available features: {len(features_june)} customers")
print(f"   Available labels: {len(labels_june)} customers")

# Create a training-ready dataset by joining features and labels
if len(features_june) > 0 and len(labels_june) > 0:
    # Inner join to get customers with both features and labels
    training_data = features_june.merge(
        labels_june[['Customer_ID', 'target', 'weighted_risk_score']], 
        on='Customer_ID', 
        how='inner'
    )
    
    print(f"\n🔗 Training-Ready Dataset:")
    print(f"   Customers with both features & labels: {len(training_data)}")
    
    if len(training_data) > 0:
        # Target analysis
        target_valid = training_data['target'].notna()
        valid_targets = training_data[target_valid]
        
        print(f"   Valid targets: {len(valid_targets)} customers")
        
        if len(valid_targets) > 0:
            target_dist = valid_targets['target'].value_counts().sort_index()
            print(f"   Target distribution:")
            for target, count in target_dist.items():
                print(f"      Target {int(target)}: {count} customers ({count/len(valid_targets):.1%})")
            
            # Check if dataset is balanced enough for modeling
            if len(target_dist) >= 2:
                minority_class = target_dist.min()
                majority_class = target_dist.max()
                imbalance_ratio = minority_class / majority_class
                
                print(f"\n📈 Class Balance Analysis:")
                print(f"   Imbalance ratio: {imbalance_ratio:.3f}")
                if imbalance_ratio >= 0.1:
                    print(f"   ✅ Reasonable balance for modeling")
                else:
                    print(f"   ⚠️  Severe imbalance - may need resampling")

print("\n" + "=" * 60)

# Feature quality analysis
print("📊 3. Feature Quality Analysis:")
print("-" * 35)

if len(training_data) > 0:
    print(f"🔍 Feature Analysis for {len(training_data)} customers:")
    
    # Exclude non-feature columns
    exclude_cols = ['Customer_ID', 'snapshot_date', 'target', 'weighted_risk_score']
    feature_columns = [col for col in training_data.columns if col not in exclude_cols]
    
    print(f"   Total features available: {len(feature_columns)}")
    
    # Missing values analysis
    print(f"\n📋 Missing Values Summary:")
    missing_summary = []
    
    for col in feature_columns:
        missing_count = training_data[col].isna().sum()
        missing_pct = missing_count / len(training_data) * 100
        missing_summary.append((col, missing_count, missing_pct))
    
    # Sort by missing percentage
    missing_summary.sort(key=lambda x: x[2], reverse=True)
    
    # Show worst missing features
    print(f"   Features with missing values:")
    has_missing = False
    for col, count, pct in missing_summary:
        if count > 0:
            print(f"      {col}: {count} missing ({pct:.1f}%)")
            has_missing = True
    
    if not has_missing:
        print(f"      ✅ No missing values in any feature!")
    
    # Feature types analysis
    print(f"\n📊 Feature Types Distribution:")
    numeric_features = training_data[feature_columns].select_dtypes(include=[np.number]).columns
    categorical_features = training_data[feature_columns].select_dtypes(include=['object']).columns
    
    print(f"   Numeric features: {len(numeric_features)}")
    print(f"   Categorical features: {len(categorical_features)}")
    
    # Sample numeric feature statistics
    if len(numeric_features) > 0:
        print(f"\n📈 Sample Numeric Feature Statistics:")
        sample_numeric = list(numeric_features)[:5]  # First 5 numeric features
        
        for feature in sample_numeric:
            stats = training_data[feature].describe()
            print(f"   {feature}: min={stats['min']:.2f}, mean={stats['mean']:.2f}, max={stats['max']:.2f}")
    
    # Sample categorical feature distributions
    if len(categorical_features) > 0:
        print(f"\n📂 Sample Categorical Feature Distributions:")
        sample_categorical = list(categorical_features)[:3]  # First 3 categorical features
        
        for feature in sample_categorical:
            value_counts = training_data[feature].value_counts()
            print(f"   {feature}: {len(value_counts)} unique values")
            # Show top categories
            for value, count in value_counts.head(3).items():
                print(f"      '{value}': {count} ({count/len(training_data):.1%})")

print("\n" + "=" * 60)

# Modeling readiness assessment
print("🎯 4. Modeling Readiness Assessment:")
print("-" * 40)

if len(training_data) > 0 and 'target' in training_data.columns:
    valid_for_modeling = training_data['target'].notna()
    modeling_data = training_data[valid_for_modeling]
    
    print(f"✅ Modeling Assessment Results:")
    print(f"   Dataset size: {len(modeling_data)} customers")
    print(f"   Feature count: {len(feature_columns)} features")
    
    if len(modeling_data) > 0:
        target_dist = modeling_data['target'].value_counts()
        min_class_size = target_dist.min()
        
        print(f"   Minimum class size: {min_class_size} samples")
        
        # Modeling readiness criteria
        criteria = {
            "Sufficient samples": len(modeling_data) >= 100,
            "Balanced classes": min_class_size >= 10,
            "Adequate features": len(feature_columns) >= 10,
            "Target available": 'target' in modeling_data.columns
        }
        
        print(f"\n📋 Modeling Readiness Checklist:")
        all_ready = True
        for criterion, passed in criteria.items():
            status = "✅" if passed else "❌"
            print(f"   {status} {criterion}")
            if not passed:
                all_ready = False
        
        if all_ready:
            print(f"\n🚀 READY FOR MODELING! Dataset meets all criteria.")
        else:
            print(f"\n⚠️  Dataset needs improvement before modeling.")
    
    # Save sample for modeling
    print(f"\n💾 Modeling Data Preview:")
    print("   First 5 customers (Customer_ID, target, first 3 features):")
    preview_cols = ['Customer_ID', 'target'] + feature_columns[:3]
    print(modeling_data[preview_cols].head())

print("\n✅ Task 3 Complete - Gold layer structure understood and modeling readiness assessed!")

🔍 Gold Layer EDA - Task 3: Understanding Gold Data Structure & Feature Analysis
📊 1. Gold Layer Data Structure Understanding:
--------------------------------------------------
🧠 Hypothesis: Features = Monthly Active Customers, Labels = Cumulative Customer History

🔍 Let's validate this understanding:

📅 2023-01-01:
   Features: 530 customers (monthly active)
   Labels: 530 customers (cumulative history)
   Overlap: 530 customers
   Features only: 0 customers
   Labels only: 0 customers
   Target distribution: 0/530 high risk (0.0%)

📅 2023-03-01:
   Features: 506 customers (monthly active)
   Labels: 1537 customers (cumulative history)
   Overlap: 506 customers
   Features only: 0 customers
   Labels only: 1031 customers
   Target distribution: 11/1537 high risk (0.7%)

📅 2023-06-01:
   Features: 517 customers (monthly active)
   Labels: 3085 customers (cumulative history)
   Overlap: 517 customers
   Features only: 0 customers
   Labels only: 2568 customers
   Target distribution: 44

In [43]:
print("🔍 Gold Layer EDA - Corrected: Cumulative Feature + Label Analysis")
print("=" * 75)

print("📊 1. Cumulative Target Distribution Analysis:")
print("-" * 50)

print("🧠 Correct Logic: For each month, use cumulative features + cumulative labels")
print("   This simulates the real modeling scenario\n")

# Corrected analysis using cumulative logic
monthly_results_corrected = []

for date in sorted(df_gold_labels['snapshot_date'].unique()):
    # Get cumulative labels for this date
    labels_cumulative = df_gold_labels[df_gold_labels['snapshot_date'] == date]
    
    # For features: get ALL features up to this date (cumulative approach)
    # This means: for each customer in labels, find their latest available features up to this date
    features_cumulative = df_gold_features[df_gold_features['snapshot_date'] <= date]
    
    if len(features_cumulative) > 0 and len(labels_cumulative) > 0:
        # For each customer in labels, get their most recent feature record up to this date
        customer_features = []
        label_customers = set(labels_cumulative['Customer_ID'])
        
        for customer_id in label_customers:
            # Get this customer's feature history up to the current date
            customer_feature_history = features_cumulative[
                features_cumulative['Customer_ID'] == customer_id
            ].sort_values('snapshot_date')
            
            if len(customer_feature_history) > 0:
                # Take the most recent feature record
                latest_features = customer_feature_history.iloc[-1]
                customer_features.append({
                    'Customer_ID': customer_id,
                    'feature_date': latest_features['snapshot_date'],
                    'has_features': True
                })
            else:
                customer_features.append({
                    'Customer_ID': customer_id, 
                    'feature_date': None,
                    'has_features': False
                })
        
        # Create feature availability dataframe
        df_feature_availability = pd.DataFrame(customer_features)
        
        # Join with labels to see modeling potential
        modeling_potential = labels_cumulative.merge(
            df_feature_availability,
            on='Customer_ID',
            how='left'
        )
        
        # Filter to customers who have both features and labels
        customers_with_both = modeling_potential[modeling_potential['has_features'] == True]
        
        if len(customers_with_both) > 0:
            # Analyze target distribution
            valid_targets = customers_with_both['target'].notna()
            target_data = customers_with_both[valid_targets]
            
            if len(target_data) > 0:
                target_dist = target_data['target'].value_counts().sort_index()
                high_risk = target_dist.get(1.0, 0)
                low_risk = target_dist.get(0.0, 0)
                total = len(target_data)
                risk_rate = high_risk / total if total > 0 else 0
                
                monthly_results_corrected.append({
                    'date': date,
                    'total_labels': len(labels_cumulative),
                    'customers_with_features': len(customers_with_both),
                    'valid_targets': total,
                    'high_risk': high_risk,
                    'low_risk': low_risk,
                    'risk_rate': risk_rate,
                    'feature_coverage': len(customers_with_both) / len(labels_cumulative)
                })
                
                print(f"📅 {date}:")
                print(f"   Total customers in labels: {len(labels_cumulative)}")
                print(f"   Customers with features: {len(customers_with_both)} ({len(customers_with_both)/len(labels_cumulative):.1%})")
                print(f"   High Risk: {high_risk} ({risk_rate:.1%}) | Low Risk: {low_risk}")
                print()

print("=" * 50)

print("📈 2. Corrected Risk Rate Progression:")
print("-" * 40)

if monthly_results_corrected:
    print("🔍 Risk rate evolution with cumulative features:")
    for result in monthly_results_corrected:
        risk_rate = result['risk_rate']
        coverage = result['feature_coverage']
        
        # Status based on both risk rate and feature coverage
        if coverage < 0.5:
            status = "⚠️  (Low feature coverage)"
        elif risk_rate == 0:
            status = "❌ (No high-risk customers)"
        elif risk_rate < 0.05:
            status = "⚠️  (Very low risk rate)"
        elif risk_rate < 0.20:
            status = "✅ (Good for modeling)"
        else:
            status = "🔥 (High risk rate)"
        
        print(f"   {result['date']}: {risk_rate:.1%} risk, {coverage:.1%} coverage {status}")

print("\n" + "=" * 50)

print("📊 3. Best Months for Modeling (Corrected Analysis):")
print("-" * 55)

# Find suitable months with corrected logic
suitable_months_corrected = []
for result in monthly_results_corrected:
    # Criteria: good feature coverage, sufficient risk samples, reasonable risk rate
    if (result['feature_coverage'] >= 0.8 and 
        result['risk_rate'] > 0.05 and 
        result['high_risk'] >= 50):  # At least 50 high-risk samples
        suitable_months_corrected.append(result)

if suitable_months_corrected:
    print("🎯 Months suitable for modeling (with corrected cumulative logic):")
    for result in suitable_months_corrected:
        print(f"   📅 {result['date']}:")
        print(f"      Customers: {result['customers_with_features']} (coverage: {result['feature_coverage']:.1%})")
        print(f"      High-risk: {result['high_risk']} ({result['risk_rate']:.1%})")
        print(f"      Low-risk: {result['low_risk']}")
        print()
    
    # Recommend best month
    best_month_corrected = max(suitable_months_corrected, 
                              key=lambda x: (x['customers_with_features'], x['feature_coverage']))
    
    print(f"🚀 RECOMMENDED TRAINING MONTH (CORRECTED): {best_month_corrected['date']}")
    print(f"   Total customers: {best_month_corrected['customers_with_features']}")
    print(f"   Feature coverage: {best_month_corrected['feature_coverage']:.1%}")
    print(f"   High-risk: {best_month_corrected['high_risk']} ({best_month_corrected['risk_rate']:.1%})")
    print(f"   Low-risk: {best_month_corrected['low_risk']}")
    
else:
    print("❌ Still no suitable months found...")
    print("   Let's check the latest month specifically:")
    
    if monthly_results_corrected:
        latest_result = monthly_results_corrected[-1]
        print(f"\n📊 Latest month ({latest_result['date']}) analysis:")
        print(f"   Total labels: {latest_result['total_labels']}")
        print(f"   With features: {latest_result['customers_with_features']}")
        print(f"   Feature coverage: {latest_result['feature_coverage']:.1%}")
        print(f"   Risk rate: {latest_result['risk_rate']:.1%}")
        print(f"   High-risk samples: {latest_result['high_risk']}")

print("\n" + "=" * 50)

print("🎯 4. Final Modeling Data Assessment:")
print("-" * 40)

if monthly_results_corrected:
    # Use the latest month for final assessment (most complete data)
    latest_month = monthly_results_corrected[-1]
    
    print(f"🔍 Final assessment using {latest_month['date']}:")
    print(f"   ✅ Total modeling customers: {latest_month['customers_with_features']}")
    print(f"   ✅ Feature coverage: {latest_month['feature_coverage']:.1%}")
    print(f"   ✅ High-risk samples: {latest_month['high_risk']}")
    print(f"   ✅ Low-risk samples: {latest_month['low_risk']}")
    print(f"   ✅ Risk rate: {latest_month['risk_rate']:.1%}")
    
    # Modeling readiness check
    modeling_ready = (
        latest_month['customers_with_features'] >= 1000 and
        latest_month['high_risk'] >= 100 and
        latest_month['feature_coverage'] >= 0.8
    )
    
    if modeling_ready:
        print(f"\n🚀 FINAL VERDICT: READY FOR MODELING!")
        print(f"   The dataset meets all requirements for machine learning")
    else:
        print(f"\n⚠️  FINAL VERDICT: Needs improvement")
        print(f"   Requirements: 1000+ customers, 100+ high-risk, 80%+ coverage")

print("\n✅ Corrected cumulative analysis complete!")

🔍 Gold Layer EDA - Corrected: Cumulative Feature + Label Analysis
📊 1. Cumulative Target Distribution Analysis:
--------------------------------------------------
🧠 Correct Logic: For each month, use cumulative features + cumulative labels
   This simulates the real modeling scenario

📅 2023-01-01 00:00:00:
   Total customers in labels: 530
   Customers with features: 530 (100.0%)
   High Risk: 0 (0.0%) | Low Risk: 530

📅 2023-02-01 00:00:00:
   Total customers in labels: 1031
   Customers with features: 1031 (100.0%)
   High Risk: 25 (2.4%) | Low Risk: 1006

📅 2023-03-01 00:00:00:
   Total customers in labels: 1537
   Customers with features: 1537 (100.0%)
   High Risk: 11 (0.7%) | Low Risk: 1526

📅 2023-04-01 00:00:00:
   Total customers in labels: 2047
   Customers with features: 2047 (100.0%)
   High Risk: 195 (9.5%) | Low Risk: 1852

📅 2023-05-01 00:00:00:
   Total customers in labels: 2568
   Customers with features: 2568 (100.0%)
   High Risk: 289 (11.3%) | Low Risk: 2279

📅 202

In [45]:
print("🔧 Gold Modeling Strategy - Fix Data Alignment")
print("=" * 60)

print("🎯 Strategy: Use Latest Labels + Historical Features")
print("-" * 50)

# Use the latest month with most comprehensive customer data
latest_date = '2024-12-01'
print(f"📅 Using {latest_date} as our modeling snapshot")

# Get all customers with labels from latest month
latest_labels = df_gold_labels[df_gold_labels['snapshot_date'] == latest_date].copy()
print(f"📊 Customers with labels: {len(latest_labels)}")

# Check target distribution
target_dist = latest_labels['target'].value_counts().sort_index()
print(f"🎯 Target distribution:")
for target, count in target_dist.items():
    if pd.notna(target):
        print(f"   Target {int(target)}: {count} customers ({count/len(latest_labels):.1%})")

print("\n" + "=" * 40)

print("🔍 Strategy 1: Find Historical Features for Label Customers")
print("-" * 55)

# For each customer in latest labels, find their historical features
# We'll look for their features from any previous month

label_customers = set(latest_labels['Customer_ID'])
print(f"🎯 Looking for features for {len(label_customers)} customers...")

# Find which customers have features in ANY month
all_feature_customers = set(df_gold_features['Customer_ID'])
customers_with_features = label_customers & all_feature_customers

print(f"✅ Customers with available features: {len(customers_with_features)}")
print(f"❌ Customers without features: {len(label_customers - all_feature_customers)}")

if len(customers_with_features) > 0:
    # For customers with features, get their most recent feature data
    feature_data_list = []
    
    for customer_id in customers_with_features:
        # Get all feature records for this customer
        customer_features = df_gold_features[df_gold_features['Customer_ID'] == customer_id]
        
        if len(customer_features) > 0:
            # Use the most recent feature record
            latest_feature = customer_features.sort_values('snapshot_date').iloc[-1]
            feature_data_list.append(latest_feature)
    
    # Create features dataframe
    df_features_for_modeling = pd.DataFrame(feature_data_list)
    
    print(f"📊 Assembled features for modeling: {len(df_features_for_modeling)} customers")
    
    # Join with labels
    modeling_data = df_features_for_modeling.merge(
        latest_labels[['Customer_ID', 'target', 'weighted_risk_score']], 
        on='Customer_ID', 
        how='inner'
    )
    
    print(f"🔗 Final modeling dataset: {len(modeling_data)} customers")
    
    if len(modeling_data) > 0:
        # Check target distribution in final dataset
        final_target_dist = modeling_data['target'].value_counts().sort_index()
        print(f"\n🎯 Final Target Distribution:")
        
        total_valid = 0
        for target, count in final_target_dist.items():
            if pd.notna(target):
                print(f"   Target {int(target)}: {count} customers ({count/len(modeling_data):.1%})")
                total_valid += count
        
        # Check if we have both classes
        if len(final_target_dist) >= 2 and total_valid > 50:
            print(f"\n🚀 SUCCESS: Dataset ready for modeling!")
            print(f"   Total customers: {len(modeling_data)}")
            print(f"   Features: {len([col for col in modeling_data.columns if col not in ['Customer_ID', 'snapshot_date', 'target', 'weighted_risk_score']])}")
            
            high_risk = final_target_dist.get(1.0, 0)
            low_risk = final_target_dist.get(0.0, 0)
            
            if high_risk >= 10 and low_risk >= 10:
                print(f"   ✅ Sufficient samples in both classes")
                
                # Save for modeling
                exclude_cols = ['Customer_ID', 'snapshot_date', 'weighted_risk_score']
                modeling_features = [col for col in modeling_data.columns if col not in exclude_cols]
                
                print(f"\n💾 Modeling Data Summary:")
                print(f"   Features: {len(modeling_features) - 1} (excluding target)")  # -1 for target column
                print(f"   Samples: {len(modeling_data)}")
                print(f"   Target column: 'target'")
                
                # Show sample
                print(f"\n📝 Sample data (first 3 rows, key columns):")
                sample_cols = ['Customer_ID', 'target'] + modeling_features[1:4]  # Show first 3 features
                print(modeling_data[sample_cols].head(3))
                
            else:
                print(f"   ⚠️  Insufficient samples - need at least 10 in each class")
        else:
            print(f"\n❌ PROBLEM: Insufficient data or missing target classes")

print("\n" + "=" * 40)

print("🔍 Strategy 2: Alternative - Use Any Month with Both Features & Labels")
print("-" * 70)

# Alternative: Find any month where we have both good features and reasonable labels
print("🔍 Checking all months for complete feature-label pairs...")

alternative_candidates = []

for date in sorted(df_gold_features['snapshot_date'].unique()):
    features_month = df_gold_features[df_gold_features['snapshot_date'] == date]
    labels_month = df_gold_labels[df_gold_labels['snapshot_date'] == date]
    
    # Join to see overlapping customers
    joined = features_month.merge(
        labels_month[['Customer_ID', 'target']], 
        on='Customer_ID', 
        how='inner'
    )
    
    if len(joined) > 0:
        valid_targets = joined['target'].notna()
        if valid_targets.sum() > 0:
            target_data = joined[valid_targets]
            high_risk_count = (target_data['target'] == 1.0).sum()
            
            if high_risk_count > 0:  # Any high-risk customers
                alternative_candidates.append({
                    'date': date,
                    'total_customers': len(target_data),
                    'high_risk': high_risk_count,
                    'low_risk': len(target_data) - high_risk_count,
                    'risk_rate': high_risk_count / len(target_data)
                })

if alternative_candidates:
    print(f"🎯 Found {len(alternative_candidates)} months with some risk data:")
    for candidate in alternative_candidates:
        print(f"   📅 {candidate['date']}: {candidate['total_customers']} customers, {candidate['high_risk']} high-risk ({candidate['risk_rate']:.1%})")
else:
    print("❌ No months found with both features and risk labels")

print("\n✅ Data alignment analysis complete!")

🔧 Gold Modeling Strategy - Fix Data Alignment
🎯 Strategy: Use Latest Labels + Historical Features
--------------------------------------------------
📅 Using 2024-12-01 as our modeling snapshot
📊 Customers with labels: 5531
🎯 Target distribution:
   Target 0: 4319 customers (78.1%)
   Target 1: 1212 customers (21.9%)

🔍 Strategy 1: Find Historical Features for Label Customers
-------------------------------------------------------
🎯 Looking for features for 5531 customers...
✅ Customers with available features: 5531
❌ Customers without features: 0
📊 Assembled features for modeling: 5531 customers
🔗 Final modeling dataset: 5531 customers

🎯 Final Target Distribution:
   Target 0: 4319 customers (78.1%)
   Target 1: 1212 customers (21.9%)

🚀 SUCCESS: Dataset ready for modeling!
   Total customers: 5531
   Features: 24
   ✅ Sufficient samples in both classes

💾 Modeling Data Summary:
   Features: 24 (excluding target)
   Samples: 5531
   Target column: 'target'

📝 Sample data (first 3 rows